In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
import module, secret_keys
from model_list import models
import pandas as pd

hf_api_key             = secret_keys.HF_TOKEN                   #<insert your own huggingface token here>
openai_api_key         = secret_keys.OPENAI_API_KEY_TEAM        #<insert your own openai token here>

In [15]:
data_pub = pd.read_csv('data_new/CT-Pub-With-Examples-Corrected.csv')
data_repo = pd.read_csv('data_new/CT-Repo-With-Examples-Processed-Version-Corrected.csv')

In [23]:
#drop these ids from both dataset 
avoid_ids = ['NCT00000620', 'NCT01483560', 'NCT04280783'] 

data_pub = data_pub[~data_pub['NCTId'].isin(avoid_ids)]
data_repo = data_repo[~data_repo['NCTId'].isin(avoid_ids)]

In [24]:
data_repo.head(3)

,NCTId,BriefTitle,EligibilityCriteria,BriefSummary,Conditions,Interventions,PrimaryOutcomes,TrialGroup,API_BaselineMeasures,API_BaselineMeasures_Corrected
1,NCT00003901,Prognostic Study of Metastases in Patients Wit...,Inclusion Criteria:\n\n1. Patient must be ≥ 18...,RATIONALE: Prognostic testing for early signs ...,"Lung Cancer,","immunohistochemistry staining method, biopsy, ...",Overall Survival in Lymph Nodes Examined Patie...,cancer,"Age, Continuous, Gender, Race/Ethnicity, Custo...","`Age, Continuous`, `Gender`, `Race/Ethnicity, ..."
2,NCT00005879,LY353381 in Preventing Breast Cancer in Women ...,DISEASE CHARACTERISTICS:\n\n* Current random f...,RATIONALE: Chemoprevention therapy is the use ...,"Breast Cancer,","arzoxifene, Placebo,","Change in Masood Score, Number of Participants...",cancer,"Age, Continuous, Sex: Female, Male, Region of ...","`Age, Continuous`, `Sex: Female, Male`, `Regio..."
3,NCT00005908,Primary Chemotherapy With Docetaxel-Capecitabi...,* INCLUSION CRITERIA:\n\nStage II or III breas...,This study will assess the usefulness of a tec...,"Breast Cancer, Breast Neoplasm,","Docetaxel - Dose A, Anastrozole, cyclophospham...","Number of Participants With Adverse Events, Ov...",cancer,"Age, Categorical, Age Continuous, Sex: Female,...","`Age, Categorical`, `Age Continuous`, `Sex: Fe..."


In [25]:
data_pub.head(2)

,NCTId,BriefTitle,EligibilityCriteria,BriefSummary,Conditions,Interventions,PrimaryOutcomes,TrialGroup,API_BaselineMeasures,API_BaselineMeasures_Corrected,Paper_BaselineMeasures,Paper_BaselineMeasures_Corrected
1,NCT00126737,Home-Based Exercise and Weight Control Program...,Inclusion Criteria:\n\n* Male \& female 50 yea...,The purpose of this study is to determine whet...,"Chronic Diseases, Obesity, Osteoarthritis, Pain,","Weight Control Nutritional Program, Home-based...","WOMAC Function, Physical Scale SF-36v, Mental ...",obesity,"Age, Continuous, Sex: Female, Male, Race/Ethni...","`Age, Continuous`, `Sex: Female, Male`, `Race/...","Age, Duration of OA, Kellgren-Lawrence Classif...","`Age`, `Duration of OA`, `Kellgren-Lawrence Cl..."
2,NCT00283686,HALT Progression of Polycystic Kidney Disease ...,Inclusion Criteria:\n\n* Diagnosis of ADPKD.\n...,The efficacy of interruption of the renin-angi...,"Kidney, Polycystic,","Lisinopril, Telmisartan, Placebo, Standard Blo...",Study A: Percent Annual Change in Total Kidney...,hypertension,"Age, Continuous, Sex: Female, Male, Race (NIH/...","`Age, Continuous`, `Sex: Female, Male`, `Race ...","Age, Weight, Height, BMI, BSA, SBP, DBP, Liver...","`Age`, `Weight`, `Height`, `BMI`, `BSA`, `SBP`..."


In [26]:
def unique_conditions_by_trialgroup(df):
    # Group the data by 'TrialGroup' and aggregate the 'Conditions' into a single string per group
    grouped_conditions = df.groupby('TrialGroup')['Conditions'].apply(lambda x: ', '.join(x.dropna())).reset_index()

    # Function to convert the comma-separated conditions into a unique set of conditions
    def get_unique_conditions(conditions_str):
        conditions_list = conditions_str.split(',')
        unique_conditions = set([condition.strip() for condition in conditions_list if condition.strip()])
        return unique_conditions

    # Apply the function and get the length of the unique condition set for each TrialGroup
    grouped_conditions['UniqueConditionCount'] = grouped_conditions['Conditions'].apply(lambda x: len(get_unique_conditions(x)))
    
    return grouped_conditions[['TrialGroup', 'UniqueConditionCount']]

In [27]:
data_pub.TrialGroup.value_counts()

TrialGroup
diabetes                  34
obesity                   18
chronic kidney disease    18
cancer                    16
hypertension              14
Name: count, dtype: int64

In [28]:
unique_conditions_by_trialgroup(data_pub)

,TrialGroup,UniqueConditionCount
0,cancer,49
1,chronic kidney disease,23
2,diabetes,39
3,hypertension,25
4,obesity,20


In [29]:
data_repo.TrialGroup.value_counts()

TrialGroup
cancer                    484
diabetes                  479
obesity                   292
hypertension              266
chronic kidney disease    169
Name: count, dtype: int64

In [31]:
unique_conditions_by_trialgroup(data_repo)

,TrialGroup,UniqueConditionCount
0,cancer,756
1,chronic kidney disease,289
2,diabetes,196
3,hypertension,188
4,obesity,205
